# 04 — Main Answer Generation & Internal Confidence Features

Bu notebook ana deterministic cevap üretimi ve içsel confidence feature çıkarımı içindir.

03 aşamasında dondurulan üç model `selected_models_v1.json` dosyasından okunur. Model isimleri,
revision'lar ve precision contract burada tekrar elle tanımlanmaz.

Her `question_id × model_id` için ana çıktı:

- `answer`
- `mean_logprob`
- `min_logprob`
- `first_token_logprob`
- `sequence_logprob_sum`
- `mean_token_entropy`
- `mean_token_entropy_normalized`
- `max_token_entropy_normalized`
- `first_token_entropy_normalized`
- `output_content_token_count`
- unknown / refusal / truncation sinyalleri
- token-level audit listeleri

Bu aşamada `reference_answer`, `accepted_aliases` ve `manual_is_correct` modele verilmez.

## Incremental cache

Cache anahtarı soru metni + model revision + quantization/precision + prompt +
generation contract + feature pipeline sürümüne bağlıdır.

Bu nedenle dataset 500 → 1000 → 2000 soruya büyüdüğünde daha önce aynı contract ile işlenen
sorular tekrar inference edilmez.

## 1. Drive ve bağımlılıklar

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip -q install "transformers==4.57.6" "accelerate>=1.3,<2" "bitsandbytes>=0.45" "huggingface_hub>=0.34"

## 2. Imports ve runtime

In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import re
import time
import unicodedata

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import accelerate
import bitsandbytes as bnb

from huggingface_hub import HfApi, login

EXPECTED_TRANSFORMERS_VERSION = "4.57.6"

if transformers.__version__ != EXPECTED_TRANSFORMERS_VERSION:
    raise RuntimeError(
        f"Beklenen transformers={EXPECTED_TRANSFORMERS_VERSION}; "
        f"mevcut={transformers.__version__}. Runtime'ı yeniden başlatın."
    )

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime gereklidir.")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA BF16 supported:", torch.cuda.is_bf16_supported())

## 3. Project paths ve validated dataset seçimi

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_Confidence_Thesis")

DATA_VALIDATED_DIR = PROJECT_ROOT / "data" / "validated"
MODEL_SELECTION_DIR = PROJECT_ROOT / "results" / "model_selection"

STAGE04_ROOT = PROJECT_ROOT / "runs" / "main_answers"
CACHE_ROOT = STAGE04_ROOT / "cache"
EXPORT_ROOT = STAGE04_ROOT / "exports"
RESULTS_STAGE04_DIR = PROJECT_ROOT / "results" / "stage04"

for p in [CACHE_ROOT, EXPORT_ROOT, RESULTS_STAGE04_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
VALIDATED_DATASET_VERSION = "validated_v002"
# ============================================================

DATASET_JSON_PATH = DATA_VALIDATED_DIR / f"{VALIDATED_DATASET_VERSION}.json"
DATASET_MANIFEST_PATH = DATA_VALIDATED_DIR / f"{VALIDATED_DATASET_VERSION}_manifest.json"
SELECTION_MANIFEST_PATH = MODEL_SELECTION_DIR / "selected_models_v1.json"

print("Dataset:", DATASET_JSON_PATH)
print("Selection:", SELECTION_MANIFEST_PATH)

## 4. Hash yardımcıları

In [ ]:
def canonical_json(obj):
    return json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def sha256_object(obj):
    return sha256_text(canonical_json(obj))

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

## 5. Validated dataset bütünlük kontrolü

In [ ]:
if not DATASET_JSON_PATH.exists():
    raise FileNotFoundError(DATASET_JSON_PATH)

if not DATASET_MANIFEST_PATH.exists():
    raise FileNotFoundError(DATASET_MANIFEST_PATH)

with open(DATASET_JSON_PATH, "r", encoding="utf-8") as f:
    validated_data = json.load(f)

with open(DATASET_MANIFEST_PATH, "r", encoding="utf-8") as f:
    dataset_manifest = json.load(f)

dataset_sha256 = file_sha256(DATASET_JSON_PATH)

if dataset_sha256 != dataset_manifest["validated_file_sha256"]:
    raise RuntimeError("Validated dataset SHA-256 manifest ile uyuşmuyor.")

if len(validated_data) != dataset_manifest["n_validated_questions"]:
    raise RuntimeError("Validated dataset soru sayısı manifest ile uyuşmuyor.")

dataset_df = pd.DataFrame(validated_data)

required_columns = {
    "question_id",
    "question",
    "answer_type",
    "category",
    "verification_status",
}

missing = required_columns - set(dataset_df.columns)
if missing:
    raise ValueError(f"Eksik dataset kolonları: {sorted(missing)}")

if dataset_df["question_id"].duplicated().any():
    raise RuntimeError("Duplicate question_id var.")

if not (dataset_df["verification_status"] == "verified").all():
    raise RuntimeError("04 yalnız verified kayıtlarla çalışır.")

print("VALIDATED DATASET OK")
print("Version:", VALIDATED_DATASET_VERSION)
print("N:", len(dataset_df))
print("SHA-256:", dataset_sha256)

## 6. Dondurulmuş model-selection manifestini yükle

In [ ]:
if not SELECTION_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "selected_models_v1.json bulunamadı. Önce 03 aşamasını tamamlayın."
    )

selection_manifest_sha256 = file_sha256(SELECTION_MANIFEST_PATH)

with open(SELECTION_MANIFEST_PATH, "r", encoding="utf-8") as f:
    selection_manifest = json.load(f)

selected_models = selection_manifest.get("selected_models", [])

if len(selected_models) < 3:
    raise RuntimeError("Tez protokolü en az üç ana model gerektiriyor.")

required_model_fields = {
    "model_id",
    "short_name",
    "revision",
    "backend",
    "trust_remote_code",
    "disable_thinking",
    "compute_dtype",
    "model_dtype",
    "run_precision_tag",
}

for spec in selected_models:
    missing = required_model_fields - set(spec.keys())
    if missing:
        raise RuntimeError(
            f"{spec.get('model_id')} model spec eksik: {sorted(missing)}"
        )

display(
    pd.DataFrame(selected_models)[
        [
            "model_id",
            "short_name",
            "revision",
            "backend",
            "compute_dtype",
            "model_dtype",
        ]
    ]
)

print("Selection manifest SHA:", selection_manifest_sha256)

## 7. Ana answer-generation contract

In [ ]:
PROMPT_VERSION = selection_manifest["prompt_version"]
GENERATION_CONFIG_VERSION = selection_manifest["generation_config_version"]

FEATURE_PIPELINE_VERSION = "internal_features_v1_content_tokens"
MAX_NEW_TOKENS = 20

SYSTEM_PROMPT = (
    "Türkçe kısa ve doğrudan olgusal cevap ver. "
    "Sadece cevabı yaz; açıklama veya gerekçe verme. "
    "Cevabı bilmiyorsan yalnızca 'Bilmiyorum.' yaz."
)

PROMPT_TEXT_SHA256 = sha256_text(SYSTEM_PROMPT)

if PROMPT_VERSION != "short_fact_tr_v1":
    raise RuntimeError(
        f"Bu notebook {PROMPT_VERSION} prompt sürümünü uygulamayı bilmiyor."
    )

print("Prompt version:", PROMPT_VERSION)
print("Generation version:", GENERATION_CONFIG_VERSION)
print("Feature version:", FEATURE_PIPELINE_VERSION)
print("Prompt SHA:", PROMPT_TEXT_SHA256)

## 8. Model-selection sorularını işaretle

In [ ]:
MODEL_SELECTION_QUESTION_IDS = set(
    selection_manifest.get("sanity_question_ids", [])
)

dataset_df["used_in_model_selection"] = (
    dataset_df["question_id"]
    .isin(MODEL_SELECTION_QUESTION_IDS)
    .astype(int)
)

print(
    "Bu dataset içindeki model-selection sorusu:",
    int(dataset_df["used_in_model_selection"].sum()),
)
print("Bu flag ileride final test split'inde exclusion olarak kullanılacak.")

## 9. Unknown / refusal yardımcıları

In [ ]:
def normalize_answer(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.replace("İ", "i").replace("I", "ı")
    text = text.lower().strip()

    text = re.sub(
        r"^[\s\.,;:!?\"'`()\[\]{}]+|[\s\.,;:!?\"'`()\[\]{}]+$",
        "",
        text,
    )
    text = re.sub(r"\s+", " ", text).strip()
    return text

PURE_UNKNOWN_FORMS = {
    "bilmiyorum",
    "emin değilim",
    "bilinmiyor",
    "bilemiyorum",
}

UNKNOWN_MARKER_PATTERNS = [
    r"\bbilmiyorum\b",
    r"\bemin değilim\b",
    r"\bbilinmiyor\b",
    r"\bbilemiyorum\b",
]

REFUSAL_PATTERNS = [
    r"\bcevap veremem\b",
    r"\byanıt veremem\b",
    r"\byanit veremem\b",
    r"\byardımcı olamam\b",
    r"\byardimci olamam\b",
    r"\bbilgi veremem\b",
]

def pure_unknown_response(answer):
    return int(normalize_answer(answer) in PURE_UNKNOWN_FORMS)

def contains_unknown_marker(answer):
    normalized = normalize_answer(answer)
    return int(
        any(re.search(pattern, normalized) for pattern in UNKNOWN_MARKER_PATTERNS)
    )

def mixed_answer_unknown(answer):
    return int(
        contains_unknown_marker(answer) == 1
        and pure_unknown_response(answer) == 0
    )

def refusal_indicator(answer):
    normalized = normalize_answer(answer)
    return int(
        any(re.search(pattern, normalized) for pattern in REFUSAL_PATTERNS)
    )

## 10. Model loader

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    Gemma3ForConditionalGeneration,
    BitsAndBytesConfig,
)

DTYPE_MAP = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}

def get_dtype(name):
    if name not in DTYPE_MAP:
        raise ValueError(f"Bilinmeyen dtype: {name}")
    return DTYPE_MAP[name]

def quant_config(model_spec):
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=get_dtype(model_spec["compute_dtype"]),
    )

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_tokenizer(frontend):
    return frontend.tokenizer if hasattr(frontend, "tokenizer") else frontend

def ensure_gated_access(model_spec):
    if model_spec["backend"] != "gemma3_processor":
        return

    api = HfApi()

    try:
        api.model_info(
            model_spec["model_id"],
            revision=model_spec["revision"],
        )
    except Exception:
        print("Gemma gated erişimi için Hugging Face login gerekiyor.")
        login()
        api.model_info(
            model_spec["model_id"],
            revision=model_spec["revision"],
        )

def load_model(model_spec):
    clear_gpu()
    ensure_gated_access(model_spec)

    model_id = model_spec["model_id"]
    revision = model_spec["revision"]
    backend = model_spec["backend"]

    print("Model:", model_id)
    print("Revision:", revision)
    print("Backend:", backend)
    print("4-bit compute:", model_spec["compute_dtype"])

    if backend == "gemma3_processor":
        frontend = AutoProcessor.from_pretrained(
            model_id,
            revision=revision,
            trust_remote_code=False,
        )

        model = Gemma3ForConditionalGeneration.from_pretrained(
            model_id,
            revision=revision,
            quantization_config=quant_config(model_spec),
            device_map="auto",
            torch_dtype=get_dtype(model_spec["model_dtype"]),
        )

    elif backend == "causal_lm":
        frontend = AutoTokenizer.from_pretrained(
            model_id,
            revision=revision,
            trust_remote_code=model_spec["trust_remote_code"],
            use_fast=True,
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            revision=revision,
            quantization_config=quant_config(model_spec),
            device_map="auto",
            trust_remote_code=model_spec["trust_remote_code"],
            torch_dtype=get_dtype(model_spec["model_dtype"]),
        )
    else:
        raise ValueError(f"Bilinmeyen backend: {backend}")

    model.eval()
    tokenizer = get_tokenizer(frontend)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    if model.generation_config.pad_token_id is None:
        model.generation_config.pad_token_id = tokenizer.pad_token_id

    print("Loaded:", model.__class__.__name__)
    print("Tokenizer:", tokenizer.__class__.__name__)
    print("EOS:", model.generation_config.eos_token_id)

    return frontend, model

## 11. Chat-template / input hazırlama

In [ ]:
def render_causal_prompt(tokenizer, question, disable_thinking=False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    if disable_thinking:
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            pass

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def build_model_inputs(frontend, model, model_spec, question):
    backend = model_spec["backend"]

    if backend == "gemma3_processor":
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}],
            },
            {
                "role": "user",
                "content": [{"type": "text", "text": question}],
            },
        ]

        inputs = frontend.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
    else:
        tokenizer = get_tokenizer(frontend)
        prompt = render_causal_prompt(
            tokenizer,
            question,
            disable_thinking=model_spec["disable_thinking"],
        )
        inputs = tokenizer(prompt, return_tensors="pt")

    return {k: v.to(model.device) for k, v in inputs.items()}

## 12. Content-token policy

EOS / PAD / diğer special tokenlar confidence feature hesaplarından çıkarılır.

03'teki logprob/entropy değerleri yalnız teknik probe idi. 04'te üretilen değerler ana deney
feature'larıdır.

In [ ]:
def eos_id_set(model):
    value = model.generation_config.eos_token_id

    if value is None:
        return set()

    if isinstance(value, int):
        return {value}

    return {int(x) for x in value}

def special_id_set(tokenizer, model):
    ids = {int(x) for x in tokenizer.all_special_ids}
    ids.update(eos_id_set(model))

    if tokenizer.pad_token_id is not None:
        ids.add(int(tokenizer.pad_token_id))

    return ids

def safe_mean(values):
    return float(np.mean(values)) if values else None

def safe_min(values):
    return float(np.min(values)) if values else None

def safe_max(values):
    return float(np.max(values)) if values else None

def decode_answer(frontend, generated_ids):
    tokenizer = get_tokenizer(frontend)
    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

## 13. Tek soru: deterministic answer + internal features

In [ ]:
@torch.inference_mode()
def infer_one_question(frontend, model, model_spec, question):
    tokenizer = get_tokenizer(frontend)

    inputs = build_model_inputs(
        frontend,
        model,
        model_spec,
        question,
    )

    prompt_len = inputs["input_ids"].shape[1]

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    generated_ids = generated.sequences[0][prompt_len:]
    answer = decode_answer(frontend, generated_ids)

    special_ids = special_id_set(tokenizer, model)

    content_token_ids = []
    content_token_strings = []
    token_logprobs = []
    token_entropies = []
    token_entropies_normalized = []

    usable_steps = min(len(generated.scores), len(generated_ids))

    for step_idx in range(usable_steps):
        token_id = int(generated_ids[step_idx].item())

        if token_id in special_ids:
            continue

        logits = generated.scores[step_idx][0].float()

        log_probs = torch.log_softmax(logits, dim=-1)
        probs = torch.softmax(logits, dim=-1)

        selected_logprob = float(log_probs[token_id].item())
        entropy = float(-(probs * log_probs).sum().item())

        vocab_size = int(logits.numel())
        entropy_normalized = float(entropy / math.log(vocab_size))

        content_token_ids.append(token_id)
        content_token_strings.append(
            tokenizer.convert_ids_to_tokens(token_id)
        )
        token_logprobs.append(selected_logprob)
        token_entropies.append(entropy)
        token_entropies_normalized.append(entropy_normalized)

    eos_ids = eos_id_set(model)

    last_generated_id = (
        int(generated_ids[-1].item())
        if len(generated_ids) > 0
        else None
    )

    ended_with_eos = int(
        last_generated_id in eos_ids
        if last_generated_id is not None
        else False
    )

    truncated_by_max_tokens = int(
        len(generated_ids) >= MAX_NEW_TOKENS
        and ended_with_eos == 0
    )

    return {
        "answer": answer,
        "generation_seconds": float(elapsed),

        "generated_token_count_total": int(len(generated_ids)),
        "output_content_token_count": int(len(content_token_ids)),
        "output_char_count": int(len(answer)),

        "sequence_logprob_sum": (
            float(np.sum(token_logprobs))
            if token_logprobs
            else None
        ),
        "mean_logprob": safe_mean(token_logprobs),
        "min_logprob": safe_min(token_logprobs),
        "first_token_logprob": (
            float(token_logprobs[0])
            if token_logprobs
            else None
        ),

        "mean_token_entropy": safe_mean(token_entropies),
        "mean_token_entropy_normalized": safe_mean(
            token_entropies_normalized
        ),
        "max_token_entropy_normalized": safe_max(
            token_entropies_normalized
        ),
        "first_token_entropy_normalized": (
            float(token_entropies_normalized[0])
            if token_entropies_normalized
            else None
        ),

        "pure_unknown_response": pure_unknown_response(answer),
        "contains_unknown_marker": contains_unknown_marker(answer),
        "mixed_answer_unknown": mixed_answer_unknown(answer),
        "refusal_indicator": refusal_indicator(answer),
        "empty_response": int(len(answer.strip()) == 0),

        "ended_with_eos": ended_with_eos,
        "truncated_by_max_tokens": truncated_by_max_tokens,

        # Audit
        "content_token_ids": content_token_ids,
        "content_token_strings": content_token_strings,
        "token_logprobs": token_logprobs,
        "token_entropies": token_entropies,
        "token_entropies_normalized": token_entropies_normalized,
    }

## 14. Config hash ve cache key

In [ ]:
def build_model_contract(model_spec):
    return {
        "model_id": model_spec["model_id"],
        "short_name": model_spec["short_name"],
        "revision": model_spec["revision"],
        "backend": model_spec["backend"],
        "trust_remote_code": model_spec["trust_remote_code"],
        "disable_thinking": model_spec["disable_thinking"],
        "quantization": {
            "load_in_4bit": True,
            "quant_type": "nf4",
            "double_quant": True,
            "compute_dtype": model_spec["compute_dtype"],
            "model_dtype": model_spec["model_dtype"],
        },
        "prompt_version": PROMPT_VERSION,
        "prompt_text_sha256": PROMPT_TEXT_SHA256,
        "generation_config_version": GENERATION_CONFIG_VERSION,
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "feature_pipeline_version": FEATURE_PIPELINE_VERSION,
        "content_token_policy": "exclude_all_special_eos_pad",
    }

def build_config_hash(model_spec):
    return sha256_object(build_model_contract(model_spec))

def build_question_text_sha(question):
    return sha256_text(unicodedata.normalize("NFKC", question))

def build_inference_cache_key(question_id, question, config_hash):
    return sha256_object(
        {
            "question_id": question_id,
            "question_text_sha256": build_question_text_sha(question),
            "config_hash": config_hash,
        }
    )

## 15. JSONL cache yardımcıları

In [ ]:
def read_jsonl(path):
    rows = []

    if not path.exists():
        return rows

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    return rows

def append_jsonl_record(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "a", encoding="utf-8") as f:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
                allow_nan=False,
            )
            + "\n"
        )
        f.flush()
        os.fsync(f.fileno())

def atomic_write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = path.with_suffix(path.suffix + ".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                    allow_nan=False,
                )
                + "\n"
            )

    os.replace(temp_path, path)

## 16. Incremental model run

Bu fonksiyon global cache'i kontrol eder; yalnız eksik sorularda model yüklenir.

Cache artifact'ına reference answer yazılmaz.

In [ ]:
def run_model_incremental(model_spec):
    model_contract = build_model_contract(model_spec)
    config_hash = build_config_hash(model_spec)
    config_short = config_hash[:12]

    short_name = model_spec["short_name"]

    model_cache_dir = CACHE_ROOT / short_name
    model_cache_dir.mkdir(parents=True, exist_ok=True)

    cache_path = model_cache_dir / f"{config_short}.jsonl"
    cache_config_path = model_cache_dir / f"{config_short}_config.json"

    if not cache_config_path.exists():
        with open(cache_config_path, "x", encoding="utf-8") as f:
            json.dump(
                {
                    "config_hash": config_hash,
                    "contract": model_contract,
                    "created_at_utc": utc_now_iso(),
                },
                f,
                ensure_ascii=False,
                indent=2,
            )
    else:
        with open(cache_config_path, "r", encoding="utf-8") as f:
            existing_config = json.load(f)

        if existing_config["config_hash"] != config_hash:
            raise RuntimeError("Cache config hash uyuşmazlığı.")

    cached_rows = read_jsonl(cache_path)
    cache_lookup = {
        row["inference_cache_key"]: row
        for row in cached_rows
    }

    required_items = []

    for _, row in dataset_df.iterrows():
        cache_key = build_inference_cache_key(
            row["question_id"],
            row["question"],
            config_hash,
        )

        required_items.append(
            {
                "question_id": row["question_id"],
                "question": row["question"],
                "question_text_sha256": build_question_text_sha(
                    row["question"]
                ),
                "answer_type": row["answer_type"],
                "category": row["category"],
                "used_in_model_selection": int(
                    row["used_in_model_selection"]
                ),
                "inference_cache_key": cache_key,
            }
        )

    pending = [
        item
        for item in required_items
        if item["inference_cache_key"] not in cache_lookup
    ]

    reused_n = len(required_items) - len(pending)

    print("\n========================================")
    print("MODEL:", model_spec["model_id"])
    print("Config hash:", config_hash)
    print("Dataset N:", len(required_items))
    print("Cache reuse:", reused_n)
    print("New inference:", len(pending))
    print("========================================")

    peak_gpu_gb = None

    if pending:
        frontend, model = load_model(model_spec)
        torch.cuda.reset_peak_memory_stats()

        for idx, item in enumerate(pending, start=1):
            result = infer_one_question(
                frontend,
                model,
                model_spec,
                item["question"],
            )

            cache_record = {
                "inference_cache_key": item["inference_cache_key"],
                "config_hash": config_hash,
                "question_id": item["question_id"],
                "question": item["question"],
                "question_text_sha256": item["question_text_sha256"],
                "model_id": model_spec["model_id"],
                "model_short_name": short_name,
                "model_revision": model_spec["revision"],
                "created_at_utc": utc_now_iso(),
                **result,
            }

            append_jsonl_record(
                cache_path,
                cache_record,
            )

            cache_lookup[
                item["inference_cache_key"]
            ] = cache_record

            print(
                f"[{idx:04d}/{len(pending):04d}] "
                f"{item['question_id']} | "
                f"{result['answer']!r} | "
                f"LP={result['mean_logprob']} | "
                f"Hn={result['mean_token_entropy_normalized']} | "
                f"{result['generation_seconds']:.2f}s"
            )

        peak_gpu_gb = float(
            torch.cuda.max_memory_allocated() / (1024 ** 3)
        )

        del model, frontend
        clear_gpu()

    else:
        print("Tüm sorular cache'de. Model yüklenmedi.")

    export_rows = []

    for item in required_items:
        cached = cache_lookup[item["inference_cache_key"]]

        export_rows.append(
            {
                "dataset_version": VALIDATED_DATASET_VERSION,
                "dataset_sha256": dataset_sha256,

                "question_id": item["question_id"],
                "question": item["question"],
                "question_text_sha256": item["question_text_sha256"],
                "answer_type": item["answer_type"],
                "category": item["category"],
                "used_in_model_selection": item["used_in_model_selection"],

                "model_id": model_spec["model_id"],
                "model_short_name": short_name,
                "model_revision": model_spec["revision"],
                "config_hash": config_hash,
                "inference_cache_key": item["inference_cache_key"],

                "answer": cached["answer"],
                "generation_seconds": cached["generation_seconds"],
                "generated_token_count_total": cached[
                    "generated_token_count_total"
                ],
                "output_content_token_count": cached[
                    "output_content_token_count"
                ],
                "output_char_count": cached["output_char_count"],

                "sequence_logprob_sum": cached["sequence_logprob_sum"],
                "mean_logprob": cached["mean_logprob"],
                "min_logprob": cached["min_logprob"],
                "first_token_logprob": cached["first_token_logprob"],

                "mean_token_entropy": cached["mean_token_entropy"],
                "mean_token_entropy_normalized": cached[
                    "mean_token_entropy_normalized"
                ],
                "max_token_entropy_normalized": cached[
                    "max_token_entropy_normalized"
                ],
                "first_token_entropy_normalized": cached[
                    "first_token_entropy_normalized"
                ],

                "pure_unknown_response": cached["pure_unknown_response"],
                "contains_unknown_marker": cached[
                    "contains_unknown_marker"
                ],
                "mixed_answer_unknown": cached["mixed_answer_unknown"],
                "refusal_indicator": cached["refusal_indicator"],
                "empty_response": cached["empty_response"],
                "ended_with_eos": cached["ended_with_eos"],
                "truncated_by_max_tokens": cached[
                    "truncated_by_max_tokens"
                ],

                "content_token_ids": cached["content_token_ids"],
                "content_token_strings": cached["content_token_strings"],
                "token_logprobs": cached["token_logprobs"],
                "token_entropies": cached["token_entropies"],
                "token_entropies_normalized": cached[
                    "token_entropies_normalized"
                ],
            }
        )

    export_dir = EXPORT_ROOT / VALIDATED_DATASET_VERSION
    export_dir.mkdir(parents=True, exist_ok=True)

    export_path = export_dir / f"{short_name}_{config_short}.jsonl"
    export_manifest_path = (
        export_dir / f"{short_name}_{config_short}_manifest.json"
    )

    if export_path.exists():
        existing_rows = read_jsonl(export_path)

        existing_keys = {
            row["inference_cache_key"]
            for row in existing_rows
        }
        expected_keys = {
            row["inference_cache_key"]
            for row in export_rows
        }

        if existing_keys != expected_keys:
            raise RuntimeError(
                "Mevcut dataset export aynı isim altında farklı key set'i taşıyor."
            )

        print("Dataset export zaten mevcut ve key set'i doğru:")
        print(export_path)

    else:
        atomic_write_jsonl(export_path, export_rows)

    export_sha = file_sha256(export_path)

    manifest = {
        "stage": "04_main_answer_internal_features",
        "dataset_version": VALIDATED_DATASET_VERSION,
        "dataset_sha256": dataset_sha256,
        "n_questions": len(export_rows),

        "model_contract": model_contract,
        "config_hash": config_hash,

        "selection_manifest_sha256": selection_manifest_sha256,

        "cache_file": str(cache_path),
        "cache_reused_this_run": reused_n,
        "new_inference_this_run": len(pending),

        "export_file": export_path.name,
        "export_file_sha256": export_sha,
        "peak_gpu_memory_gb_for_new_inference": peak_gpu_gb,

        "runtime": {
            "gpu": torch.cuda.get_device_name(0),
            "python": platform.python_version(),
            "pytorch": torch.__version__,
            "transformers": transformers.__version__,
            "accelerate": accelerate.__version__,
            "bitsandbytes": bnb.__version__,
        },

        "created_at_utc": utc_now_iso(),
    }

    if export_manifest_path.exists():
        with open(export_manifest_path, "r", encoding="utf-8") as f:
            existing_manifest = json.load(f)

        if existing_manifest["export_file_sha256"] != export_sha:
            raise RuntimeError("Export manifest SHA uyuşmuyor.")
    else:
        with open(export_manifest_path, "x", encoding="utf-8") as f:
            json.dump(
                manifest,
                f,
                ensure_ascii=False,
                indent=2,
            )

    print("\nSTAGE-04 MODEL EXPORT TAMAMLANDI")
    print("Export:", export_path)
    print("Manifest:", export_manifest_path)

    return {
        "model_id": model_spec["model_id"],
        "config_hash": config_hash,
        "export_path": export_path,
        "manifest_path": export_manifest_path,
        "n": len(export_rows),
    }

## 17. Model index tablosu

In [ ]:
MODEL_INDEX_TABLE = pd.DataFrame(
    [
        {
            "index": idx,
            "model_id": spec["model_id"],
            "short_name": spec["short_name"],
            "revision": spec["revision"][:12],
            "compute_dtype": spec["compute_dtype"],
        }
        for idx, spec in enumerate(selected_models)
    ]
)

display(MODEL_INDEX_TABLE)

## 18. Tek modeli çalıştır

Tablodaki index'i kullanın. Aynı hücreyi daha sonra yeni validated dataset sürümünde tekrar
çalıştırmak güvenlidir; cache yalnız eksik soruları üretir.

In [ ]:
MODEL_INDEX_TO_RUN = 0

model_spec_to_run = selected_models[MODEL_INDEX_TO_RUN]

run_result = run_model_incremental(
    model_spec_to_run
)

run_result

Üç model için yukarıdaki index'i sırayla değiştirin.

Aynı modeli yanlışlıkla tekrar çalıştırmanız sorun değildir; cache nedeniyle yeni inference
yapılmaz.

## 19. Üç model export'unu yükle

In [ ]:
def expected_export_for_model(model_spec):
    config_hash = build_config_hash(model_spec)
    config_short = config_hash[:12]

    return (
        EXPORT_ROOT
        / VALIDATED_DATASET_VERSION
        / f"{model_spec['short_name']}_{config_short}.jsonl"
    )

combined_rows = []
missing_exports = []

for spec in selected_models:
    export_path = expected_export_for_model(spec)

    if not export_path.exists():
        missing_exports.append(str(export_path))
        continue

    combined_rows.extend(read_jsonl(export_path))

if missing_exports:
    print("Eksik model export'ları:")
    for item in missing_exports:
        print("-", item)

    raise RuntimeError("Önce üç modelin stage-04 run'ını tamamlayın.")

combined_df = pd.DataFrame(combined_rows)

print("Toplam model-answer:", len(combined_df))
print(
    "Beklenen:",
    len(dataset_df) * len(selected_models),
)

## 20. Coverage ve uniqueness

In [ ]:
expected_rows = len(dataset_df) * len(selected_models)

if len(combined_df) != expected_rows:
    raise RuntimeError(
        f"Beklenen {expected_rows}, mevcut {len(combined_df)} satır."
    )

if combined_df.duplicated(
    subset=["question_id", "model_id"]
).any():
    raise RuntimeError("Duplicate question_id × model_id var.")

coverage = (
    combined_df
    .groupby("model_id")
    .agg(
        n=("question_id", "count"),
        unique_questions=("question_id", "nunique"),
        empty_rate=("empty_response", "mean"),
        pure_unknown_rate=("pure_unknown_response", "mean"),
        contains_unknown_rate=("contains_unknown_marker", "mean"),
        mixed_answer_unknown_rate=("mixed_answer_unknown", "mean"),
        refusal_rate=("refusal_indicator", "mean"),
        truncation_rate=("truncated_by_max_tokens", "mean"),
        mean_output_tokens=("output_content_token_count", "mean"),
        mean_generation_seconds=("generation_seconds", "mean"),
    )
    .reset_index()
)

display(coverage)

COVERAGE_PATH = (
    RESULTS_STAGE04_DIR
    / f"{VALIDATED_DATASET_VERSION}_coverage.csv"
)

coverage.to_csv(COVERAGE_PATH, index=False)
print(COVERAGE_PATH)

## 21. Internal feature audit

In [ ]:
feature_columns = [
    "mean_logprob",
    "min_logprob",
    "first_token_logprob",
    "sequence_logprob_sum",
    "mean_token_entropy",
    "mean_token_entropy_normalized",
    "max_token_entropy_normalized",
    "first_token_entropy_normalized",
]

audit_rows = []

for model_id, group in combined_df.groupby("model_id"):
    row = {
        "model_id": model_id,
        "n": len(group),
    }

    for feature in feature_columns:
        numeric = pd.to_numeric(
            group[feature],
            errors="coerce",
        )

        row[f"{feature}_missing"] = int(
            numeric.isna().sum()
        )

        finite = np.isfinite(
            numeric.dropna()
        )

        row[f"{feature}_nonfinite"] = int(
            (~finite).sum()
        )

    audit_rows.append(row)

feature_audit_df = pd.DataFrame(audit_rows)
display(feature_audit_df)

FEATURE_AUDIT_PATH = (
    RESULTS_STAGE04_DIR
    / f"{VALIDATED_DATASET_VERSION}_internal_feature_audit.csv"
)

feature_audit_df.to_csv(
    FEATURE_AUDIT_PATH,
    index=False,
)

print(FEATURE_AUDIT_PATH)

## 22. Şüpheli output audit

In [ ]:
suspicious_mask = (
    (combined_df["empty_response"] == 1)
    | (combined_df["mixed_answer_unknown"] == 1)
    | (combined_df["refusal_indicator"] == 1)
    | (combined_df["truncated_by_max_tokens"] == 1)
)

suspicious_df = (
    combined_df.loc[
        suspicious_mask,
        [
            "question_id",
            "model_id",
            "question",
            "answer",
            "empty_response",
            "pure_unknown_response",
            "mixed_answer_unknown",
            "refusal_indicator",
            "truncated_by_max_tokens",
        ],
    ]
    .sort_values(["model_id", "question_id"])
)

print("Şüpheli model-answer:", len(suspicious_df))
display(suspicious_df.head(100))

## 22B. Stage-03 → Stage-04 deterministic reproducibility audit

03'te model-selection için kullanılan 30 immutable sanity sorusu, 04 dataset'inin içinde de
bulunuyorsa aynı:

- model revision
- prompt version
- generation config
- quantization / precision contract
- `do_sample=False`

altında **aynı answer text** üretmelidir.

04'te feature hesaplama yöntemi değişmiştir; fakat generation contract değişmemiştir.

Bu nedenle bu audit yalnız `answer` metnini karşılaştırır. Logprob/entropy değerleri
karşılaştırılmaz; 03 değerleri teknik probe, 04 değerleri content-token ana feature'larıdır.

Bir mismatch varsa 04'ü dondurmadan önce incelenmelidir.

In [ ]:
MODEL_SELECTION_RUNS_DIR = (
    PROJECT_ROOT
    / "runs"
    / "model_selection"
    / selection_manifest["dataset_version"]
)

repro_rows = []
missing_stage03_runs = []

for spec in selected_models:
    stage03_path = (
        MODEL_SELECTION_RUNS_DIR
        / (
            f"{spec['short_name']}_"
            f"{spec['revision'][:8]}"
            f"{spec['run_precision_tag']}_"
            f"{PROMPT_VERSION}_"
            f"{GENERATION_CONFIG_VERSION}.jsonl"
        )
    )

    if not stage03_path.exists():
        missing_stage03_runs.append(
            str(stage03_path)
        )
        continue

    stage03_rows = read_jsonl(
        stage03_path
    )

    stage03_df = pd.DataFrame(
        stage03_rows
    )

    stage04_model_df = combined_df.loc[
        combined_df["model_id"]
        == spec["model_id"]
    ].copy()

    overlap = (
        stage03_df[
            [
                "question_id",
                "question",
                "answer",
            ]
        ]
        .rename(
            columns={
                "question": "question_stage03",
                "answer": "answer_stage03",
            }
        )
        .merge(
            stage04_model_df[
                [
                    "question_id",
                    "question",
                    "answer",
                ]
            ].rename(
                columns={
                    "question": "question_stage04",
                    "answer": "answer_stage04",
                }
            ),
            on="question_id",
            how="inner",
            validate="one_to_one",
        )
    )

    overlap["question_text_match"] = (
        overlap["question_stage03"]
        == overlap["question_stage04"]
    ).astype(int)

    overlap["answer_exact_match"] = (
        overlap["answer_stage03"]
        == overlap["answer_stage04"]
    ).astype(int)

    overlap["model_id"] = spec[
        "model_id"
    ]

    repro_rows.extend(
        overlap.to_dict(
            orient="records"
        )
    )

if missing_stage03_runs:
    print(
        "Stage-03 run dosyası bulunamayan modeller:"
    )
    for item in missing_stage03_runs:
        print("-", item)

    raise RuntimeError(
        "Reproducibility audit tamamlanamadı."
    )

repro_df = pd.DataFrame(
    repro_rows
)

expected_overlap = (
    len(
        selection_manifest[
            "sanity_question_ids"
        ]
    )
    *
    len(
        selected_models
    )
)

if len(repro_df) != expected_overlap:
    raise RuntimeError(
        f"Beklenen overlap={expected_overlap}; "
        f"bulunan={len(repro_df)}."
    )

if not (
    repro_df[
        "question_text_match"
    ] == 1
).all():
    bad_questions = repro_df.loc[
        repro_df[
            "question_text_match"
        ] == 0
    ]

    display(
        bad_questions
    )

    raise RuntimeError(
        "03 ve 04 arasında aynı question_id için "
        "question text değişmiş."
    )

repro_summary = (
    repro_df
    .groupby(
        "model_id"
    )
    .agg(
        n_overlap=(
            "question_id",
            "count",
        ),
        exact_answer_matches=(
            "answer_exact_match",
            "sum",
        ),
        exact_answer_match_rate=(
            "answer_exact_match",
            "mean",
        ),
    )
    .reset_index()
)

display(
    repro_summary
)

REPRO_AUDIT_PATH = (
    RESULTS_STAGE04_DIR
    / (
        f"{VALIDATED_DATASET_VERSION}"
        "_stage03_stage04_reproducibility.csv"
    )
)

repro_df.to_csv(
    REPRO_AUDIT_PATH,
    index=False,
)

mismatch_df = repro_df.loc[
    repro_df[
        "answer_exact_match"
    ] == 0,
    [
        "question_id",
        "model_id",
        "question_stage04",
        "answer_stage03",
        "answer_stage04",
    ],
]

print(
    "Toplam exact-answer mismatch:",
    len(
        mismatch_df
    ),
)

if not mismatch_df.empty:
    display(
        mismatch_df
    )

    raise RuntimeError(
        "03 → 04 deterministic answer mismatch bulundu. "
        "Stage 04'ü dondurmayın; mismatch'leri inceleyin."
    )

print(
    "STAGE-03 → STAGE-04 REPRODUCIBILITY BAŞARILI"
)
print(
    REPRO_AUDIT_PATH
)

## 23. Stage-04 completion manifest

In [ ]:
stage04_exports = []

for spec in selected_models:
    config_hash = build_config_hash(spec)
    export_path = expected_export_for_model(spec)

    manifest_path = export_path.with_name(
        export_path.stem + "_manifest.json"
    )

    if not export_path.exists() or not manifest_path.exists():
        raise RuntimeError(
            f"Eksik stage-04 artifact: {spec['model_id']}"
        )

    stage04_exports.append(
        {
            "model_id": spec["model_id"],
            "model_revision": spec["revision"],
            "config_hash": config_hash,
            "export_file": str(export_path),
            "export_sha256": file_sha256(export_path),
            "manifest_file": str(manifest_path),
        }
    )

STAGE04_COMPLETION_PATH = (
    RESULTS_STAGE04_DIR
    / f"{VALIDATED_DATASET_VERSION}_stage04_complete.json"
)

stage04_completion = {
    "stage": "04_main_answer_internal_features",
    "dataset_version": VALIDATED_DATASET_VERSION,
    "dataset_sha256": dataset_sha256,
    "n_questions": len(dataset_df),
    "n_models": len(selected_models),
    "n_model_answer_rows": len(combined_df),

    "selection_manifest": str(SELECTION_MANIFEST_PATH),
    "selection_manifest_sha256": selection_manifest_sha256,

    "prompt_version": PROMPT_VERSION,
    "prompt_text_sha256": PROMPT_TEXT_SHA256,
    "generation_config_version": GENERATION_CONFIG_VERSION,
    "feature_pipeline_version": FEATURE_PIPELINE_VERSION,

    "model_exports": stage04_exports,
    "coverage_file": str(COVERAGE_PATH),
    "feature_audit_file": str(FEATURE_AUDIT_PATH),
    "reproducibility_audit_file": str(REPRO_AUDIT_PATH),
    "stage03_stage04_exact_answer_match_required": True,

    "created_at_utc": utc_now_iso(),
}

if STAGE04_COMPLETION_PATH.exists():
    with open(
        STAGE04_COMPLETION_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        existing = json.load(f)

    if existing["dataset_sha256"] != dataset_sha256:
        raise RuntimeError(
            "Mevcut stage04 completion manifest farklı dataset SHA taşıyor."
        )

    print("Stage-04 completion manifest zaten mevcut:")
    print(STAGE04_COMPLETION_PATH)

else:
    with open(
        STAGE04_COMPLETION_PATH,
        "x",
        encoding="utf-8",
    ) as f:
        json.dump(
            stage04_completion,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print("STAGE 04 DONDURULDU:")
    print(STAGE04_COMPLETION_PATH)

# 04 tamamlanma kriterleri

- [ ] validated dataset checksum doğrulandı
- [ ] `selected_models_v1.json` okundu
- [ ] üç model exact revision ile çalıştırıldı
- [ ] Gemma/Qwen/Phi precision contract korundu
- [ ] reference answer modele verilmedi
- [ ] deterministic ana cevap üretildi
- [ ] special/EOS/PAD tokenlar confidence hesaplarından çıkarıldı
- [ ] logprob feature'ları üretildi
- [ ] raw + normalized entropy feature'ları üretildi
- [ ] unknown / refusal / truncation sinyalleri kaydedildi
- [ ] token-level audit listeleri saklandı
- [ ] global incremental cache oluşturuldu
- [ ] dataset-specific export'lar üretildi
- [ ] `question_id × model_id` coverage tam
- [ ] feature audit tamamlandı
- [ ] 03 → 04 deterministic reproducibility audit %100 exact-answer match verdi
- [ ] `validated_vXXX_stage04_complete.json` oluşturuldu

## Sonraki aşama — 05 Self-Confidence & Self-Verification

05 ana cevabı tekrar üretmeyecek. 04'teki frozen answer'ları okuyup ayrı sorgularla:

- self-reported confidence
- verification A/B/C
- verification probability sinyalleri
- verification margin
- verification entropy

üretecek.